# Fully Sharded Data Parallel (FSDP) with Ray Train

This notebook will walk you through a high level overview of FSDP with Ray Train.

<div class="alert alert-block alert-info">

Here is the roadmap for this notebook:

<ol>
  <li>What is FSDP?</li>
  <li>FSDP vs DDP simplified</li>
  <li>How to use FSDP with Ray Train?</li>
</ol>
</div>

**Imports**

In [ ]:
import os
import tempfile
from typing import Any

import torch
import ray.train.torch
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    StateDictType,
    FullStateDictConfig,
)
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.models import resnet18
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor, Normalize, Compose
from torch.distributed.fsdp import ShardingStrategy

## What is FSDP?

Fully Sharded Data Parallel (FSDP) is a parallelism method that combines the advantages of data and model parallelism for distributed training.

### FSDP Workflow

Below is a diagram (taken and adapted from PyTorch) that shows the FSDP workflow

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/FSDP.png" width="800">


Here is a table explaining the different phases, their steps and what happens:

<table>
  <thead>
    <tr>
      <th>Phase</th>
      <th>Step</th>
      <th>What Happens</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Initialization</td>
      <td>Parameter sharding</td>
      <td>Each rank stores only its own shard of every parameter it owns.</td>
    </tr>
    <tr>
      <td>Forward pass</td>
      <td>
        <ol>
          <li>all_gather</li>
          <li>Compute</li>
          <li>Free full weights</li>
        </ol>
      </td>
      <td>Ranks gather one another’s shards to reconstruct full weights(parameters), execute the forward computation, then immediately free the temporary shards.</td>
    </tr>
    <tr>
      <td>Backward pass</td>
      <td>
        <ol>
          <li>all_gather</li>
          <li>Back-propagate</li>
          <li>reduce_scatter</li>
          <li>Free full weights</li>
        </ol>
      </td>
      <td>Shards are re-gathered, gradients are computed, then reduced-and-scattered so each rank keeps only its own gradient shard; full replicas are discarded again.</td>
    </tr>
  </tbody>
</table>


## FSDP vs DDP simplified

Let's go over a toy example  (inspired by [this guide on parallelism from huggingface](https://huggingface.co/docs/transformers/v4.13.0/en/parallelism)) comparing how FSDP and DDP operate.

### 1  Toy model

| **La** | **Lb** | **Lc** |
|:------:|:------:|:------:|
 a₀ | b₀ | c₀ |
 a₁ | b₁ | c₁ |
 a₂ | b₂ | c₂ |

Layer `La` contains the weights `[a₀, a₁, a₂]`

Total parameters = **9 scalars** (3 per layer).

---

### 2  Parameter layout at rest  

#### 2.1  DDP  (full replication)

| **GPU** | **La**           | **Lb**           | **Lc**           |
|---------|------------------|------------------|------------------|
| 0       | a₀ a₁ a₂         | b₀ b₁ b₂         | c₀ c₁ c₂         |
| 1       | a₀ a₁ a₂         | b₀ b₁ b₂         | c₀ c₁ c₂         |
| 2       | a₀ a₁ a₂         | b₀ b₁ b₂         | c₀ c₁ c₂         |

*Each worker stores **100 %** of the model (parameters + optimizer states + gradients).*

---

#### 2.2  FSDP  (parameter sharding)

| **GPU** | **La** | **Lb** | **Lc** |
|---------|:------:|:------:|:------:|
| 0       | a₀     | b₀     | c₀     |
| 1       | a₁     | b₁     | c₁     |
| 2       | a₂     | b₂     | c₂     |

*At rest each worker keeps only **1∕3** of every layer—so memory ≈ 33 % of DDP.*

---

### 3  Execution flow (GPU 0’s perspective)

#### 3.1  FSDP

<details>
<summary><strong>Forward pass (Layer La)</strong></summary>

1. **Need:** a₀ a₁ a₂  
2. **Has locally:** a₀  
3. **Gather:** a₁ from GPU 1, a₂ from GPU 2  
4. **Compute:** *y = La(x)* with full weights  
5. **Free:** a₁, a₂ (optional halfway-release)

Repeat for **Lb** then **Lc**.
</details>

<details>
<summary><strong>Backward pass (Layer Lc)</strong></summary>

1. **Need:** c₀ c₁ c₂  
2. **Has locally:** c₀  
3. **Gather:** c₁ from GPU 1, c₂ from GPU 2  
4. **Compute:** ∂L/∂c₀, ∂L/∂c₁, ∂L/∂c₂  
5. **Reduce-scatter:** average gradients; GPU k keeps only its own shard (cₖ)  
6. **Free:** c₁, c₂ (optional halfway-release)

Work upstream through **Lb → La** in reverse order.
</details>

<details>
<summary><strong>Optimizer step</strong></summary>

*Purely local.*  
GPU 0 updates a₀, b₀, c₀ using the averaged gradient shards already resident in memory. No extra communication.
</details>

---

#### 3.2  DDP  (for comparison)

| Phase             | What GPU 0 already owns         | Communication |
|-------------------|---------------------------------|---------------|
| **Forward**       | Full La, Lb, Lc                 | None          |
| **Backward**      | Full La, Lb, Lc + grads         | **All-reduce**|
| **Optimizer step**| Full params & full grads        | None          |

---

### 4  Key take-aways

|                         | **DDP**                           | **FSDP**                               |
|-------------------------|-----------------------------------|----------------------------------------|
| **Memory footprint**    | Replicated (× #GPUs)              | 1∕#GPUs at rest; slightly higher during gather |
| **Communication**       | All-reduce once per layer (backward) | Gather + reduce-scatter per layer      |
| **Optimizer states**    | Fully replicated                  | Sharded—1∕#GPUs memory|
| **Implementation ease** | Very simple                       | More knobs (wrapping policy, offloading,prefetch) |
| **When to prefer**      | Fits in memory; small models      | Large models that would otherwise OOM  |

> **Rule of thumb:**  
> *If you can replicate the model comfortably, DDP wins on simplicity and sometimes speed.  
> If you’re out of memory or pushing model scale, FSDP (or ZeRO-style sharding) is the way forward.*

---

#### 5  Why we wrapped each layer separately

For illustration we treated **“one layer = one FSDP unit.”**  
In practice you can:

* **Cluster layers** into larger FSDP units to reduce the number of gather calls.
* **Wrap only the largest sub-modules** (e.g., big embeddings, attention blocks) and leave tiny layers unsharded.

Experiment with *auto-wrap policies* to find the sweet spot between memory savings and communication overhead.


## When to Consider FSDP

- **Model no longer fits** on a single GPU even with mixed precision.  
- **Batch size is GPU memory-bound** under classic DDP.  
- **You have multiple GPUs** with sufficient interconnect bandwidth.  
- **You already use DDP** but need to push to larger architectures (e.g., multi-billion-parameter transformers).  
- **You want minimal code changes**—wrap layers with `torch.distributed.fsdp.FullyShardedDataParallel`.  

FSDP lets you step beyond the memory limits of traditional data parallelism while keeping your training loop largely unchanged.

## How to use FSDP with Ray Train?

Let's go over a simple example of how to use FSDP with Ray Train and PyTorch.

Below is a sample training function that we will use to train our model.

In [ ]:
def train_func(config):
    # Model, Loss, Optimizer
    model = resnet18(num_classes=10)
    model.conv1 = torch.nn.Conv2d(
        1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    )

    # [1] Prepare the model in FSDP.
    model = prepare_model_with_fsdp(model, fsdp_kwargs=config["fsdp_kwargs"])

    criterion = CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    # [2] Load FSDP model and optimizer state if a checkpoint is found.
    loaded_checkpoint = ray.train.get_checkpoint()
    if loaded_checkpoint:
        load_fsdp_checkpoint(model, optimizer, loaded_checkpoint)

    # Data
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    data_dir = os.path.join(tempfile.gettempdir(), "data")
    train_data = FashionMNIST(
        root=data_dir, train=True, download=True, transform=transform
    )
    train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
    train_loader = ray.train.torch.prepare_data_loader(train_loader)

    world_rank = ray.train.get_context().get_world_rank()

    # Training
    running_loss = 0.0          # accumulate loss over batches
    num_batches  = 0            # count batches for averaging
    for epoch in range(2):
        if ray.train.get_context().get_world_size() > 1:
            train_loader.sampler.set_epoch(epoch)

        for images, labels in train_loader:
            # This is done by `prepare_data_loader`!
            # images, labels = images.to("cuda"), labels.to("cuda")
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # track running statistics
            running_loss += loss.item()
            num_batches  += 1

        # [3] Report metrics and FSDP checkpoint.
        avg_loss = running_loss / num_batches  # mean loss over all batches
        metrics = {"loss": avg_loss, "epoch": epoch}
        report_metrics_and_save_fsdp_checkpoint(model, optimizer, metrics)

        if world_rank == 0:
            print(metrics)

    # Save model for inference
    save_model_for_inference(model, world_rank)

### Prepare the model

To prepare the model, we can either directly use the `FullyShardedDataParallel` class from PyTorch or use the RayTrain `train.torch.prepare_model` function.

In [ ]:
def prepare_model_with_fsdp(model: torch.nn.Module, fsdp_kwargs: Any) -> FSDP:
    return ray.train.torch.prepare_model(
        model, parallel_strategy="fsdp", parallel_strategy_kwargs=fsdp_kwargs
    )

Here is a table of the main keyword arguments for FSDP:

| Parameter | What it controls | Typical values & when to use |
|-----------|-----------------|------------------------------|
| `sharding_strategy` | Where model parameters, gradients, and optimizer state live across workers | **`FULL_SHARD`** (default) – shard everything for maximum memory savings.<br>**`SHARD_GRAD_OP`** – replicate parameters but shard gradients/optimizer state; useful when parameters fit but gradients do not.<br>**`NO_SHARD`** – disable sharding (behaves like DDP); handy for debugging mixed setups. |
| `cpu_offload` | Whether inactive parameter shards are moved to CPU RAM | **`offload_params=False`** (default) – fastest, keeps data on GPU.<br>**`offload_params=True`** – frees GPU memory at the cost of extra PCIe traffic; enables larger models on small-RAM GPUs. |
| `auto_wrap_policy` | Decides which sub-modules get wrapped (and therefore sharded) | **Default `None`** – only the root module is wrapped; developers must manually wrap sub-modules.<br>**`size_based_auto_wrap_policy`** – wrap any sub-tree whose parameters exceed a threshold (e.g., 100 M), balancing memory savings and communication cost.<br>**`transformer_auto_wrap_policy`** – targets common Transformer block patterns.<br>**Custom callable or `ModuleWrapPolicy`** – fine-grained control (e.g., wrap embeddings and attention layers but skip layer norms). |


#### Save FSDP checkpoint

Here is a table of the different `state_dict_type` for FSDP checkpoints

| Option | What each rank writes to disk | Memory / I-O cost | Best use-case |
|--------|------------------------------|-------------------|---------------|
| **`FULL_STATE_DICT`** | Entire model collected on **rank 0** (others can be empty if `rank0_only=True`) | • Highest RAM on rank 0<br>• One big file<br>• Extra all-gather traffic | Exporting for single-GPU inference, sharing with users who are **not** using FSDP, or swapping to DDP. |
| **`LOCAL_STATE_DICT`** | Only the local parameter + optimizer shards that the rank already owns (plain `Tensor`s) | • Low per-rank RAM<br>• One file **per rank**<br>• No gather | Frequent in-training checkpoints when you will resume with **the same world-size and FSDP topology**. |
| **`SHARDED_STATE_DICT`** | Local shards stored as `ShardedTensor`s **plus metadata** describing the full tensor layout | • Similar footprint to LOCAL<br>• One file **per rank**<br>• Metadata lets FSDP reshape later | Large-model training where you may change world-size (e.g., 8 → 16 GPUs) or want safer, self-describing shards. |

**During training** prefer **LOCAL** or **SHARDED** every _N_ steps/epochs to avoid the costly gather required by `FULL_STATE_DICT`.

In [ ]:
def report_metrics_and_save_fsdp_checkpoint(
    model: FSDP, optimizer: torch.optim.Optimizer, metrics: dict
) -> None:
    with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
        with FSDP.state_dict_type(model, StateDictType.SHARDED_STATE_DICT):
            rank = ray.train.get_context().get_world_rank()

            torch.save(
                model.state_dict(),
                os.path.join(temp_checkpoint_dir, f"model-shard-rank={rank=}.pt"),
            )
            torch.save(
                FSDP.optim_state_dict(model, optim=optimizer),
                os.path.join(temp_checkpoint_dir, f"optim-shard-rank={rank=}.pt"),
            )

        checkpoint = ray.train.Checkpoint.from_directory(temp_checkpoint_dir)

        ray.train.report(metrics, checkpoint=checkpoint)

For **final export** switch once to `FULL_STATE_DICT`, often with `CPU offload` and `rank0_only=True`, so only rank 0 keeps the big tensor in RAM:


In [ ]:
def save_model_for_inference(model: FSDP, world_rank: int) -> None:
    with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
        with FSDP.state_dict_type(
            model,
            StateDictType.FULL_STATE_DICT,
            FullStateDictConfig(
                rank0_only=True, offload_to_cpu=False
            ),  # potentially can offload to CPU too
        ):
            state_dict = model.state_dict()
            checkpoint = None

            if world_rank == 0:
                torch.save(
                    state_dict,
                    os.path.join(temp_checkpoint_dir, "full-model.pt"),
                )
                checkpoint = ray.train.Checkpoint.from_directory(
                    temp_checkpoint_dir
                )

            ray.train.report({}, checkpoint=checkpoint, checkpoint_dir_name="full_model")


### Load FSDP checkpoint

In [ ]:
def load_fsdp_checkpoint(
    model: FSDP, optimizer: torch.optim.Optimizer, ckpt: ray.train.Checkpoint
) -> tuple[
    FSDP,
    torch.optim.Optimizer,
]:
    rank = ray.train.get_context().get_world_rank()

    with ckpt.as_directory() as checkpoint_dir:
        model_state_shard = torch.load(
            os.path.join(checkpoint_dir, f"model-shard-rank={rank=}.pt")
        )
        optim_state_shard = torch.load(
            os.path.join(checkpoint_dir, f"optim-shard-rank={rank=}.pt")
        )

        with FSDP.state_dict_type(model, StateDictType.SHARDED_STATE_DICT):
            model.load_state_dict(model_state_shard)
            optim_state_dict = FSDP.optim_state_dict_to_load(
                model, optimizer, optim_state_shard
            )
            optimizer.load_state_dict(optim_state_dict)

    return model, optimizer

### Run the distributed training

In [ ]:
# [4] Configure scaling and resource requirements.
scaling_config = ray.train.ScalingConfig(num_workers=2, use_gpu=True, resources_per_worker={"accelerator_type:T4": 0.0001})

# [5] Launch distributed training job.
trainer = ray.train.torch.TorchTrainer(
    train_func,
    scaling_config=scaling_config,
    train_loop_config={
        "fsdp_kwargs": {
            "sharding_strategy": ShardingStrategy.FULL_SHARD,
        }
    },
    # [5a] If running in a multi-node cluster, this is where you
    # should configure the run's persistent storage that is accessible
    # across all worker nodes.
    run_config=ray.train.RunConfig(
        storage_path="/mnt/cluster_storage/",
        name="fsdp_mnist",
        failure_config=ray.train.FailureConfig(max_failures=2),
    ),
)
result = trainer.fit()

#### Load checkpoint for inference

In [ ]:
model = resnet18(num_classes=10)
model.conv1 = torch.nn.Conv2d(
    1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
)
model_state_shard = torch.load("/mnt/cluster_storage/fsdp_mnist/full_model/full-model.pt", map_location='cpu')

model.load_state_dict(model_state_shard)

In [ ]:
transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
test_data = FashionMNIST(
    root=".", train=False, download=True, transform=transform
)
test_data

In [ ]:
test_data.data[0].reshape(1, 1, 28, 28).float()

In [ ]:
model.eval()
with torch.no_grad():
    out = model(test_data.data[0].reshape(1, 1, 28, 28).float())
    predicted_label = out.argmax().item()
    test_label = test_data.targets[0].item()
    print(f"{predicted_label=} {test_label=}")

## Profiling DDP vs FSDP



Run the following command to profile the memory usage of a DDP run:

In [ ]:
#!python code/example_train_profile_memory.py --num_workers 2 --distribution_strategy ddp 

Run the following command to prove training will OOM without FSDP.

In [ ]:
# !python code/example_train_profile_memory.py --num_workers 1 --hidden_dim 3840

Run the following command to shard the

In [ ]:
# !python code/example_train_profile_memory.py --num_workers 2 --hidden_dim 3840 --distribution_strategy fsdp